# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and analyze the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is published with a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Install mlcroissant if needed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully!\n")
print(f"Name: {metadata.name}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print("\nDescription:")
print(metadata.description)

## 2. Data Overview
List available record sets, their `@id` values, and the fields (columns) within each. These names and `@id`s are used for referencing when loading records for analysis.

In [ ]:
# Print a structured overview of record sets, fields, and columns by their @id
def print_record_sets_overview(ds):
    print("Available Record Sets:\n-------------------")
    for rs in ds.metadata.record_sets:
        print(f"Record Set: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields (columns):")
        for field in rs.fields:
            print(f"    - {field.name} (id: {field.id}, dtype: {getattr(field, 'data_type', 'Unknown')})")
        print()

print_record_sets_overview(dataset)

## 3. Data Extraction
Let's extract records from each record set and store them in pandas DataFrames for analysis. We use their `@id` as the reference.

_**Note:** For this dataset, the main record set typically contains protein-level measurements._

In [ ]:
# List the available record sets by their @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

print(f"Record set IDs found: {record_set_ids}")

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"  - Columns: {list(df.columns)} (first 3 shown): {list(df.columns)[:3]}")
    print(f"  - Loaded {len(df)} records.")
    dataframes[record_set_id] = df

# For demonstration, pick the first record set for subsequent analysis
main_rs = record_set_ids[0] if record_set_ids else None
df = dataframes.get(main_rs)

print("\nSample of the main record set DataFrame:")
if df is not None:
    display(df.head())
else:
    print("No data loaded. Check if the schema contains record sets.")

## 4. Exploratory Data Analysis (EDA)
Let's clean and explore the main DataFrame:

- Filter by peptide counts (or similar numeric field)
- Normalize the numeric field
- Group by another key field (e.g., 'accession' or 'description' if available)

Be sure to reference fields **by their `@id`**.

In [ ]:
# Identify common field IDs likely for abundance or peptide counts
numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]
print("Numeric fields detected:", numeric_fields)

# For this dataset, a typical measure is 'cr:peptideCount' or 'cr:abundance'
numeric_field = None
for candidate in ['cr:peptideCount', 'cr:abundance', 'cr:molecularWeight', 'cr:coverage']:
    if candidate in df.columns:
        numeric_field = candidate
        break
if numeric_field is None and numeric_fields:
    numeric_field = numeric_fields[0]

if numeric_field is not None:
    print(f"Selected numeric field for filtering: {numeric_field}")

    # Drop NA values
    filtered_df = df[df[numeric_field].notna()]
    threshold = filtered_df[numeric_field].mean()  # Example threshold: mean

    filtered_strict = filtered_df[filtered_df[numeric_field] > threshold]

    print(f"Filtered '{numeric_field}' > {threshold:.2f} (N={len(filtered_strict)} records):")
    display(filtered_strict.head())

    # Normalize
    filtered_strict[f"{numeric_field}_normalized"] = (
        filtered_strict[numeric_field] - filtered_strict[numeric_field].mean()
    ) / filtered_strict[numeric_field].std()

    print(f"\nNormalized {numeric_field} (z-score):")
    display(filtered_strict[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by categorical (if available)
    candidate_groups = ['cr:accession', 'cr:description', 'cr:sampleLabel']
    group_field = None
    for c in candidate_groups:
        if c in filtered_strict.columns:
            group_field = c
            break

    if group_field:
        grouped = filtered_strict.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of {numeric_field} by {group_field} (first 5):")
        display(grouped.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and any group-wise aggregated values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and df is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group field was found and has reasonable cardinality, plot group means
    if 'grouped' in locals() and hasattr(grouped, 'head') and grouped.size < 100:
        plt.figure(figsize=(12,3))
        grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We demonstrated the use of `mlcroissant` to:
- Ingest Croissant-based dataset metadata
- Enumerate record sets and fields using `@id`
- Load and explore records as pandas DataFrames
- Filter and normalize quantitative results (e.g., peptide counts or abundance)
- Visualize numeric field distributions

**Further analyses can now leverage these data structures for proteomics, machine learning, and publication-ready plots.**